# NN_05 — walk-forward

`BUILD_SPEC.md` §7, row 05: fit per cycle on the rolling window, score the next, collect per-cycle IC. Acceptance: **an IC series over the same cycles as `bundle.ic_result`**, paired cycle by cycle against the LightGBM baseline stored there.

**This notebook reports the raw walk-forward output and claims nothing beyond it.** `NN_06` is the decisive FiLM ablation; a win or loss against LightGBM is not concluded here.

**Three protocol decisions, recorded here (2026-09-08), fixed for `NN_05`–`NN_08`:**

1. **Training objective — a deliberate correction.** `NN_04` implemented the literal `BUILD_SPEC.md` §8 target `softmax(2**y − 1)` and measured that on this data, in float32, it puts all of its mass on the 148 grade-9 names and exactly zero on grades 0–8: a top-decile-only objective that undercuts a full-list ranking experiment. The measured walk-forward therefore uses a **graded ListNet target** built from the gain scaled to `[0, 1]`: `gain = (2**y − 1) / (2**9 − 1)`, so every grade contributes to the ordering.
2. **Target sharpness — a fixed, non-tuned temperature.** The first executed version of this notebook used `target = softmax(gain)` (temperature 1.0) and measured that it is too close to uniform: entropy 7.2436 against log N = 7.3085, at most ~0.065 nats of attainable improvement per list, and early stopping selecting step 10–50 on all 45 fits, so the scored models were near-init. The active protocol is therefore `target = softmax(gain / 0.5)`: **temperature 0.5, set once, used for every cycle, every seed, and every later comparison (`NN_06`–`NN_08`)**. It is a protocol decision recorded up front, not a hyperparameter searched over; no other value was tried. The temperature-1.0 numbers are kept below only as superseded context. The target's actual per-grade mass and entropy at both temperatures are printed below.
3. **Device — CPU.** `NN_04` measured that same-seed training on MPS is not bit-reproducible (about 10⁻⁴ nats run-to-run) while CPU is. Every measured fit here runs on CPU so that each (cycle, seed) result is exactly reproducible; the runtime is printed per fit.

**Protocol.** For each target cycle the baseline trained, the production `training_dataset.build_training_dataset` builds the rolling window; its most recent included quarter is the early-stopping **validation** quarter and the remaining included quarters are the training quarters. Full-batch AdamW (lr 1e-3, weight decay 1e-2), at most 1,000 steps, validation loss every 10 steps, patience 20 checks, best-validation weights restored before the target cycle is scored. Five seeds (`nn_common.SEEDS`). The NN is scored on **exactly the names the baseline's IC was measured on** — the baseline's Materials-excluded ranked candidates with a computable forward return — and every IC is `multi_factor_ranking_runner.spearman_correlation`.

**Data mode: REAL DATA.** Everything below is computed from the cached real-data slice (`rd.slice_bundle()`, fingerprint verified in `NN_00`) plus the real daily closes from `rd.load_raw()`, which the forward returns of the earlier measured cycles require (the slice's price cache covers only the last three cycles). No synthetic, simulated, or approximated value appears anywhere; if the raw exports are absent, every computing cell prints the explicit unavailability notice and does nothing.

Model code lives **in this notebook**, identical to `NN_04`'s except for the loss target. Kernel: `atlasquant-venv` (= `.venv/bin/python`).

In [1]:
import nn_common as nc

rd = nc.rd
AVAILABLE = nc.provenance_banner("NN_05_walkforward.ipynb")

neural_ranker / NN_05_walkforward.ipynb
DATA MODE: REAL DATA.

Source: Bloomberg CSV exports under this repository's gitignored raw-data
root for multi_factor_ranking_ml -- 1,520 tickers, quarterly fundamentals
from 1980 onward, plus daily closes and macro series.

Every number below is computed from that real data through production
AtlasQuant code. No synthetic, simulated, imputed, or approximated values
are used anywhere in this notebook series.

Bounded slice: quarterly evaluation cycles 2020-10-01 .. 2024-01-01
(a genuine subset of the history, chosen so each notebook runs quickly --
not a downsample or an approximation of a longer run).

SOURCE MANIFEST (SHA-256 of every raw CSV the pipeline reads)
  raw root: /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
  fundamentals_quarterly.csv      281.3 MB  2026-08-31 23:01:10  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
  Close_Price.csv                  67.6 MB  2026-08-31 23:07:42  2723e5

libraries         : python=3.14.5, torch=2.13.0, numpy=2.5.2, pandas=3.0.5, scipy=1.18.0, lightgbm=4.7.0


## The model and the corrected objective

The architecture is `BUILD_SPEC.md` §8 exactly as built in `NN_04`. Only the ListNet target differs, as stated above.

In [2]:
import copy
import time
from datetime import datetime, time as _time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

N_FEATURES = len(nc.STOCK_CONTINUOUS)   # 63
N_MACRO = len(nc.MACRO)                 # 6
SECTOR_VOCAB, SECTOR_DIM = len(nc.SECTOR_VOCABULARY), 8
QUARTER_VOCAB, QUARTER_DIM = 4, 4
HIDDEN, LATENT = 128, 64
N_GRADES = rd.config().n_relevance_grades   # 10
TEMPERATURE = 0.5   # fixed protocol decision (see the top of this notebook); never tuned, shared by NN_06-NN_08


class NeuralRanker(nn.Module):
    def __init__(self, use_film: bool = True, dropout: float = 0.3):
        super().__init__()
        self.use_film = use_film
        self.sector_emb = nn.Embedding(SECTOR_VOCAB, SECTOR_DIM)
        self.quarter_emb = nn.Embedding(QUARTER_VOCAB, QUARTER_DIM)
        self.encoder = nn.Sequential(
            nn.Linear(2 * N_FEATURES + SECTOR_DIM + QUARTER_DIM, HIDDEN),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(HIDDEN),
            nn.Linear(HIDDEN, LATENT),
            nn.GELU(),
        )
        self.film = nn.Sequential(nn.Linear(N_MACRO, 32), nn.GELU(), nn.Linear(32, 2 * LATENT))
        nn.init.normal_(self.film[-1].weight, std=1e-4)
        nn.init.zeros_(self.film[-1].bias)
        self.head = nn.Sequential(nn.Linear(LATENT, 32), nn.GELU(), nn.Linear(32, 1))

    def film_params(self, m):
        gb = self.film(m)
        return gb[:LATENT], gb[LATENT:]

    def forward(self, X, mask, sector, quarter, m):
        z = torch.cat([X, mask, self.sector_emb(sector), self.quarter_emb(quarter - 1)], dim=1)
        h = self.encoder(z)
        if self.use_film:
            gamma, beta = self.film_params(m)
            h = (1.0 + gamma) * h + beta
        return self.head(h).squeeze(-1)


def scaled_gain(y):
    # NN_05 correction: NDCG gain scaled to [0, 1], so every grade carries mass under the softmax
    return (2.0 ** y.to(torch.float32) - 1.0) / (2.0 ** (N_GRADES - 1) - 1.0)


def graded_target(y, temperature=TEMPERATURE):
    return torch.softmax(scaled_gain(y) / temperature, dim=0)


def listnet_loss(scores, y):
    return -(graded_target(y) * F.log_softmax(scores, dim=0)).sum()


def score_cycle(net, inputs):
    return net(inputs.X, inputs.mask, inputs.sector, inputs.quarter, inputs.m)

In [3]:
if rd.require_data():
    bundle = rd.slice_bundle()
    y_demo = torch.tensor([row.relevance for row in bundle.labeled_quarters[bundle.cycles[-2].quarter_start]])
    p_active = graded_target(y_demo)                       # temperature 0.5, the active protocol
    p_superseded = graded_target(y_demo, temperature=1.0)  # temperature 1.0, superseded
    literal = torch.softmax(2.0 ** y_demo.to(torch.float32) - 1.0, dim=0)   # NN_04's literal spec target
    entropy = lambda q: float(-(q[q > 0] * torch.log(q[q > 0])).sum())
    print(f"target mass per grade on {bundle.cycles[-2].quarter_start} (N = {len(y_demo)}):")
    print(f"{'grade':<7}{'names':>6}{'T = 0.5 (active)':>18}{'T = 1.0 (superseded)':>22}{'literal (NN_04)':>17}")
    for g in range(N_GRADES):
        sel = y_demo == g
        print(f"{g:<7}{int(sel.sum()):>6}{float(p_active[sel].sum()):>18.4f}{float(p_superseded[sel].sum()):>22.4f}{float(literal[sel].sum()):>17.4f}")
    e_active, e_superseded, e_literal = entropy(p_active), entropy(p_superseded), entropy(literal)
    print(f"\nentropy (loss floor): T = 0.5 {e_active:.4f}   T = 1.0 {e_superseded:.4f}   literal {e_literal:.4f}   log N = {np.log(len(y_demo)):.4f}")
    print(f"attainable improvement per list (log N - entropy): T = 0.5 {np.log(len(y_demo)) - e_active:.4f} nats   T = 1.0 {np.log(len(y_demo)) - e_superseded:.4f} nats")
    print(f"T = 0.5 entropy below T = 1.0 entropy by {e_superseded - e_active:.4f} nats")
    nonzero = lambda q: int(sum(float(q[y_demo == g].sum()) > 0 for g in range(N_GRADES)))
    print("grades with non-zero mass: T = 0.5", nonzero(p_active), "of 10; T = 1.0", nonzero(p_superseded), "of 10; literal", nonzero(literal), "of 10")
    assert nonzero(p_active) == N_GRADES, "every grade must receive non-zero target mass"
    assert e_active < e_superseded, "the sharpened target must have lower entropy than the temperature-1.0 target"
    assert TEMPERATURE == 0.5

target mass per grade on 2023-10-01 (N = 1493):
grade   names  T = 0.5 (active)  T = 1.0 (superseded)  literal (NN_04)
0         167            0.0584                0.0870           0.0000
1         147            0.0516                0.0767           0.0000
2         147            0.0520                0.0770           0.0000
3         148            0.0532                0.0781           0.0000
4         147            0.0545                0.0788           0.0000
5         147            0.0580                0.0813           0.0000
6         148            0.0662                0.0872           0.0000
7         147            0.0845                0.0982           0.0000
8         147            0.1394                0.1261           0.0000
9         148            0.3823                0.2095           1.0000

entropy (loss floor): T = 0.5 6.9843   T = 1.0 7.2436   literal 4.9972   log N = 7.3085
attainable improvement per list (log N - entropy): T = 0.5 0.3242 nats   T = 1.0 0

## The baseline's scoring universe and real forward returns

`bundle.ic_result` stores each trained cycle's Materials-excluded ranked candidates and its IC, but not the forward returns the IC was computed from — those come from daily closes, and the slice caches prices only for the last three cycles. `rd.load_raw()` therefore loads the complete real price history once. The baseline IC is then **recomputed** from its stored ranked candidates, the production `build_forward_return_outcome`, and `spearman_correlation`; it must equal the stored IC exactly, which proves the pairing universe and returns used below are the baseline's own.

In [4]:
if rd.require_data():
    from atlas_quant.backtest.multi_factor_ranking_runner import decile_spread, spearman_correlation
    from atlas_quant.strategies.multi_factor_ranking_ml.forward_return import build_forward_return_outcome
    from atlas_quant.strategies.multi_factor_ranking_ml.model_training import TrainingState

    IC_CALLS = 0

    def ic_of(scores, returns):
        # the only IC function used anywhere in this notebook
        global IC_CALLS
        IC_CALLS += 1
        return spearman_correlation(list(scores), list(returns))

    t0 = time.perf_counter()
    data = rd.load_raw()
    print(f"load_raw: {time.perf_counter() - t0:.1f}s; {len(data.prices_by_instrument)} instruments with prices")

    cycles = bundle.cycles
    baseline = {r.cycle.quarter_start: r for r in bundle.ic_result.cycle_results}
    TRAINED = [r.cycle for r in bundle.ic_result.cycle_results if r.training_state == TrainingState.TRAINED]
    MEASURED = [r.cycle for r in bundle.ic_result.cycle_results if r.ic is not None]
    print("trained cycles ", len(TRAINED), [c.quarter_start.isoformat() for c in TRAINED])
    print("measured cycles", len(MEASURED), [c.quarter_start.isoformat() for c in MEASURED])

    UNIVERSE = {}   # quarter_start -> (instrument ids, clipped forward returns, baseline scores), in the baseline's ranked order
    print(f"\n{'cycle':<12}{'ranked':>7}{'scored':>7}{'stored':>8}{'stored IC':>11}{'recomputed':>12}{'equal':>7}{'Materials':>10}{'rejected':>9}")
    for i, cycle in enumerate(cycles):
        r = baseline[cycle.quarter_start]
        if r.ic is None:
            continue
        nxt = cycles[i + 1]
        exit_cutoff = datetime.combine(nxt.quarter_start, _time.min)
        obs_by_id = {o.instrument_id: o for o in bundle.feature_results[cycle.quarter_start].observations}
        ids, rets, base_scores = [], [], []
        for ranked in r.decision_summary.ranked_candidates:
            outcome = build_forward_return_outcome(
                ranked.instrument_id, cycle.quarter_start, cycle.quarter_start, nxt.quarter_start,
                data.prices_by_instrument.get(ranked.instrument_id, ()), exit_cutoff,
            )
            if outcome.clipped_return is not None:
                ids.append(ranked.instrument_id); rets.append(outcome.clipped_return); base_scores.append(ranked.score)
        recomputed = ic_of(base_scores, rets)
        materials = sum(1 for x in ids if obs_by_id[x].sector == "Materials")
        rejected = len(r.decision_summary.rejected_candidates)
        print(f"{cycle.quarter_start.isoformat():<12}{len(r.decision_summary.ranked_candidates):>7}{len(ids):>7}{r.scored_for_ic_count:>8}"
              f"{r.ic:>+11.4f}{recomputed:>+12.4f}{str(recomputed == r.ic):>7}{materials:>10}{rejected:>9}")
        assert recomputed == r.ic, "recomputed baseline IC must equal the stored IC bit for bit"
        assert len(ids) == r.scored_for_ic_count and materials == 0
        assert all(x in obs_by_id for x in ids) and len(set(ids)) == len(ids)
        UNIVERSE[cycle.quarter_start] = (ids, rets, base_scores)
    print("\nbaseline IC recomputed bit-for-bit from stored ranked candidates + real prices on all", len(UNIVERSE), "measured cycles: True")

load_raw: 59.8s; 1520 instruments with prices
trained cycles  9 ['2022-01-01', '2022-04-01', '2022-07-01', '2022-10-01', '2023-01-01', '2023-04-01', '2023-07-01', '2023-10-01', '2024-01-01']
measured cycles 8 ['2022-01-01', '2022-04-01', '2022-07-01', '2022-10-01', '2023-01-01', '2023-04-01', '2023-07-01', '2023-10-01']

cycle        ranked scored  stored  stored IC  recomputed  equal Materials rejected


2022-01-01     1396   1370    1370    -0.0261     -0.0261   True         0       72


2022-04-01     1397   1372    1372    -0.1900     -0.1900   True         0       72


2022-07-01     1400   1374    1374    +0.0618     +0.0618   True         0       72


2022-10-01     1402   1376    1376    +0.0583     +0.0583   True         0       72


2023-01-01     1406   1381    1381    +0.0708     +0.0708   True         0       72


2023-04-01     1407   1382    1382    +0.0335     +0.0335   True         0       72


2023-07-01     1415   1391    1391    +0.0877     +0.0877   True         0       73


2023-10-01     1420   1401    1401    +0.1240     +0.1240   True         0       73

baseline IC recomputed bit-for-bit from stored ranked candidates + real prices on all 8 measured cycles: True


## Tensors on CPU

Every cycle's tensors go through the `NN_02` policy and the `NN_04` fixed input transform, on CPU. Every model input must be finite after the `0.0` placeholder step, and the mask must still equal the original `NaN` locations.

In [5]:
if rd.require_data():
    DEVICE = torch.device("cpu")
    ALL = nc.bundle_cycle_tensors(bundle, transform=nc.normalize_cycle)
    MI = {q: nc.to_model_input(t, DEVICE) for q, t in ALL.items()}
    print("device used:", DEVICE, "| torch threads:", torch.get_num_threads())
    total_replaced = 0
    for q, inputs in MI.items():
        for name in ("X", "mask", "sector", "quarter", "m"):
            assert bool(torch.isfinite(getattr(inputs, name)).all()), f"{q} {name} not finite"
        if inputs.y is not None:
            assert bool(torch.isfinite(inputs.y).all())
        assert np.array_equal(inputs.mask.numpy().astype(bool), np.isnan(ALL[q].X)), f"{q}: mask != original NaN locations"
        assert inputs.n_nan_replaced == int(ALL[q].mask.sum())
        total_replaced += inputs.n_nan_replaced
    print(f"{len(MI)} cycles on {DEVICE}: all inputs finite, mask == original NaN locations, {total_replaced:,} placeholder entries in total")

device used: cpu | torch threads: 8
14 cycles on cpu: all inputs finite, mask == original NaN locations, 25,684 placeholder entries in total


## The walk-forward

For each cycle the baseline trained, the production `build_training_dataset` determines the rolling window (its row counts must match the per-quarter tensors exactly, so what is trained on is the production training set row for row). The most recent included quarter is held out for early stopping; the rest are trained on, one ListNet list per quarter, loss averaged over quarters. Before any fit, every training and validation quarter is asserted to be strictly before the target and to have its labels available strictly before the target's training cutoff, and the validation quarter is asserted to sit inside the trailing window.

In [6]:
if rd.require_data():
    from atlas_quant.strategies.multi_factor_ranking_ml.config import FEATURE_SCHEMA_VERSION
    from atlas_quant.strategies.multi_factor_ranking_ml.training_dataset import (
        build_training_dataset,
        check_training_eligibility,
    )

    cfg = rd.config()
    LR, WEIGHT_DECAY = 1e-3, 1e-2
    MAX_STEPS, EVAL_EVERY, PATIENCE = 1000, 10, 20

    def training_window(target):
        cutoff = datetime.combine(target.cutoff, _time.min)
        dataset = build_training_dataset(
            target.quarter_start, cutoff, bundle.labeled_quarters,
            strategy_id=cfg.strategy_id, feature_schema_version=FEATURE_SCHEMA_VERSION,
            ml_train_years=cfg.ml_train_years, model_config_identity=cfg.model.identity(),
        )
        assert check_training_eligibility(dataset, min_train_quarters=cfg.min_train_quarters).eligible
        included = list(dataset.included_quarters)
        assert dataset.groups == tuple(ALL[q].n for q in included), "training-set rows must match the per-quarter tensors"
        assert dataset.instrument_ids == tuple(i for q in included for i in ALL[q].instrument_ids)
        val_q, train_qs = included[-1], included[:-1]
        window_start = target.quarter_start.replace(year=target.quarter_start.year - cfg.ml_train_years)
        # point-in-time assertions, per target cycle
        for q in train_qs + [val_q]:
            assert q < target.quarter_start, "a training/validation quarter is not before the target"
            assert ALL[q].label_available_at < cutoff, "a label would not be available at the target's cutoff"
            assert q >= window_start, "a quarter lies outside the trailing window"
        assert val_q == max(included) and target.quarter_start not in included
        return train_qs, val_q, cutoff

    def fit_and_score(target, seed):
        train_qs, val_q, cutoff = training_window(target)
        nc.seed_everything(seed)
        net = NeuralRanker(use_film=True).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        best_val, best_state, best_step, bad, steps = float("inf"), None, 0, 0, 0
        t0 = time.perf_counter()
        for step in range(1, MAX_STEPS + 1):
            net.train()
            opt.zero_grad()
            loss = torch.stack([listnet_loss(score_cycle(net, MI[q]), MI[q].y) for q in train_qs]).mean()
            loss.backward()
            opt.step()
            steps = step
            if step % EVAL_EVERY == 0:
                net.eval()
                with torch.no_grad():
                    val = listnet_loss(score_cycle(net, MI[val_q]), MI[val_q].y).item()
                if val < best_val:
                    best_val, best_state, best_step, bad = val, copy.deepcopy(net.state_dict()), step, 0
                else:
                    bad += 1
                    if bad >= PATIENCE:
                        break
        net.load_state_dict(best_state)
        net.eval()
        with torch.no_grad():
            scores_all = score_cycle(net, MI[target.quarter_start]).numpy()
        assert scores_all.shape == (ALL[target.quarter_start].n,) and np.isfinite(scores_all).all()
        record = {
            "seed": seed, "target": target.quarter_start.isoformat(),
            "train_quarters": [q.isoformat() for q in train_qs], "validation_quarter": val_q.isoformat(),
            "steps": steps, "best_step": best_step, "best_val_loss": best_val, "final_train_loss": loss.item(),
            "scored_count": 0, "ic": None, "decile_spread": None, "seconds": time.perf_counter() - t0,
        }
        if target.quarter_start in UNIVERSE:
            ids, rets, _ = UNIVERSE[target.quarter_start]
            position = {iid: k for k, iid in enumerate(ALL[target.quarter_start].instrument_ids)}
            nn_scores = [float(scores_all[position[i]]) for i in ids]
            assert len(nn_scores) == len(ids) == baseline[target.quarter_start].scored_for_ic_count
            record["scored_count"] = len(nn_scores)
            record["ic"] = ic_of(nn_scores, rets)
            record["decile_spread"] = decile_spread(list(zip(nn_scores, rets)))
            record["scores"] = nn_scores
        return record

    RESULTS = []
    print(f"device {DEVICE} | AdamW lr {LR} wd {WEIGHT_DECAY} | max {MAX_STEPS} steps, validate every {EVAL_EVERY}, patience {PATIENCE} checks | seeds {nc.SEEDS}")
    print(f"{'target':<12}{'seed':>5}{'train q':>8}{'validation':>12}{'steps':>6}{'best@':>6}{'best val':>10}{'train':>9}{'scored':>7}{'IC':>9}{'sec':>6}")
    for target in TRAINED:
        for seed in nc.SEEDS:
            rec = fit_and_score(target, seed)
            RESULTS.append(rec)
            ic_txt = f"{rec['ic']:+.4f}" if rec["ic"] is not None else "    -   "
            print(f"{rec['target']:<12}{seed:>5}{len(rec['train_quarters']):>8}{rec['validation_quarter']:>12}{rec['steps']:>6}{rec['best_step']:>6}"
                  f"{rec['best_val_loss']:>10.5f}{rec['final_train_loss']:>9.5f}{rec['scored_count']:>7}{ic_txt:>9}{rec['seconds']:>6.1f}")
    total_seconds = sum(r["seconds"] for r in RESULTS)
    print(f"\nfits: {len(RESULTS)} ({len(TRAINED)} cycles x {len(nc.SEEDS)} seeds); total {total_seconds:.0f}s; mean {total_seconds / len(RESULTS):.1f}s per fit")
    for target in TRAINED:
        secs = sum(r["seconds"] for r in RESULTS if r["target"] == target.quarter_start.isoformat())
        print(f"  {target.quarter_start}: {secs:.1f}s for {len(nc.SEEDS)} fits")

device cpu | AdamW lr 0.001 wd 0.01 | max 1000 steps, validate every 10, patience 20 checks | seeds (0, 1, 2, 3, 4)
target       seed train q  validation steps best@  best val    train scored       IC   sec


2022-01-01      0       3  2021-07-01   220    20   7.28509  7.02690   1370  +0.0732   2.2


2022-01-01      1       3  2021-07-01   220    20   7.28336  7.03380   1370  +0.0613   2.3


2022-01-01      2       3  2021-07-01   220    20   7.29062  7.02230   1370  +0.0783   2.3


2022-01-01      3       3  2021-07-01   220    20   7.28209  7.03285   1370  +0.0998   2.2


...

2024-01-01      0      11  2023-07-01   230    30   7.28033  7.12838      0     -      8.9


2024-01-01      1      11  2023-07-01   230    30   7.27463  7.12661      0     -      8.9


2024-01-01      2      11  2023-07-01   220    20   7.27750  7.12871      0     -      8.6


2024-01-01      3      11  2023-07-01   240    40   7.27432  7.12648      0     -      9.4


2024-01-01      4      11  2023-07-01   230    30   7.27171  7.12621      0     -      9.0

fits: 45 (9 cycles x 5 seeds); total 243s; mean 5.4s per fit
  2022-01-01: 11.3s for 5 fits
  2022-04-01: 14.8s for 5 fits
  2022-07-01: 19.3s for 5 fits
  2022-10-01: 21.1s for 5 fits
  2023-01-01: 25.8s for 5 fits
  2023-04-01: 30.4s for 5 fits
  2023-07-01: 34.3s for 5 fits
  2023-10-01: 41.4s for 5 fits
  2024-01-01: 44.9s for 5 fits


### Determinism on CPU

The first trained cycle, seed 0, fitted again: scores, IC and losses must be bit-identical to the run above.

In [7]:
if rd.require_data():
    first = RESULTS[0]
    again = fit_and_score(TRAINED[0], nc.SEEDS[0])
    same = (
        again["ic"] == first["ic"] and again["best_val_loss"] == first["best_val_loss"]
        and again["final_train_loss"] == first["final_train_loss"] and again["scores"] == first["scores"]
        and again["best_step"] == first["best_step"] and again["steps"] == first["steps"]
    )
    print(f"{TRAINED[0].quarter_start} seed {nc.SEEDS[0]} repeated on {DEVICE}: IC {first['ic']:+.6f} vs {again['ic']:+.6f}, "
          f"best val {first['best_val_loss']:.6f} vs {again['best_val_loss']:.6f}, scores bit-identical: {again['scores'] == first['scores']}")
    print("bit-identical repeat:", same)
    assert same

2022-01-01 seed 0 repeated on cpu: IC +0.073248 vs +0.073248, best val 7.285092 vs 7.285092, scores bit-identical: True
bit-identical repeat: True


### Early-stopping diagnostic — validation quarter only, feeds no reported number

Every fit above selected its best validation loss within the first 50 steps, at either temperature. To show what that means rather than guess, seed 0 is re-fitted for every target cycle **without early stopping, to the 1,000-step cap**, recording at every 10th step the validation loss, the Spearman correlation between the validation quarter's scores and its relevance grades (an in-window ranking-quality proxy — the validation quarter's labels were available before the target cutoff, so this looks at nothing after it), and the training loss. No target-cycle score or IC is computed here, and nothing in this cell changes the protocol or any reported number.

In [8]:
if rd.require_data():
    CHECK_AT = (10, 20, 30, 50, 100, 200, 300, 500, 1000)
    diag = {}
    t0 = time.perf_counter()
    for target in TRAINED:
        train_qs, val_q, cutoff = training_window(target)
        nc.seed_everything(nc.SEEDS[0])
        net = NeuralRanker(use_film=True).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        val_grades = MI[val_q].y.numpy().tolist()
        rows = []
        for step in range(1, MAX_STEPS + 1):
            net.train()
            opt.zero_grad()
            loss = torch.stack([listnet_loss(score_cycle(net, MI[q]), MI[q].y) for q in train_qs]).mean()
            loss.backward()
            opt.step()
            if step % EVAL_EVERY == 0:
                net.eval()
                with torch.no_grad():
                    val_scores = score_cycle(net, MI[val_q])
                    val_loss = listnet_loss(val_scores, MI[val_q].y).item()
                rows.append((step, val_loss, spearman_correlation(val_scores.numpy().tolist(), val_grades), loss.item()))
        diag[target.quarter_start] = rows
    print(f"diagnostic: seed {nc.SEEDS[0]}, {len(diag)} cycles, {MAX_STEPS} steps each without early stopping, {time.perf_counter() - t0:.0f}s")
    print(f"{'step':>6}{'val loss, mean':>16}{'val rho(score, grade), mean':>29}{'train loss, mean':>18}")
    for step in CHECK_AT:
        at = [r for rows in diag.values() for r in rows if r[0] == step]
        print(f"{step:>6}{np.mean([r[1] for r in at]):>16.5f}{np.mean([r[2] for r in at]):>29.4f}{np.mean([r[3] for r in at]):>18.5f}")
    print()
    print(f"{'cycle':<12}{'val loss min':>13}{'@step':>7}{'val rho max':>13}{'@step':>7}{'rho at loss-min step':>22}{'rho at step 1000':>18}")
    for q, rows in diag.items():
        loss_min = min(rows, key=lambda r: r[1])
        rho_max = max(rows, key=lambda r: r[2])
        print(f"{q.isoformat():<12}{loss_min[1]:>13.5f}{loss_min[0]:>7}{rho_max[2]:>13.4f}{rho_max[0]:>7}{loss_min[2]:>22.4f}{rows[-1][2]:>18.4f}")
    rho_at = lambda step: np.mean([r[2] for rows in diag.values() for r in rows if r[0] == step])
    print(f"\nmean validation rho(score, grade): step 20 {rho_at(20):+.4f}, step 100 {rho_at(100):+.4f}, step 500 {rho_at(500):+.4f}, step 1000 {rho_at(1000):+.4f}")

diagnostic: seed 0, 9 cycles, 1000 steps each without early stopping, 228s
  step  val loss, mean  val rho(score, grade), mean  train loss, mean
    10         7.28927                       0.0596           7.25884
    20         7.28736                       0.0628           7.24636
    30         7.29150                       0.0490           7.23571
    50         7.30342                       0.0285           7.21629
   100         7.34980                       0.0266           7.16099
   200         7.40859                       0.0398           7.09775
   300         7.41777                       0.0491           7.06980
   500         7.43446                       0.0513           7.04219
  1000         7.44498                       0.0430           7.01802

cycle        val loss min  @step  val rho max  @step  rho at loss-min step  rho at step 1000
2022-01-01        7.28509     20       0.0988    390                0.0348            0.0821
2022-04-01        7.31494     20      

## Superseded context: the temperature-1.0 run

The first executed version of this notebook (2026-09-08, identical protocol except `target = softmax(gain)` at temperature 1.0) produced, on these same 8 cycles: NN seed-mean IC +0.0445 (std 0.0752, hit rate 0.875), ensemble-of-seeds +0.0478, paired NN − LightGBM mean +0.0170 (std 0.0547), NN higher in 6 of 8 cycles — with the best validation step at 10–50 on every one of the 45 fits and every fit stopped by patience at step 210–250. Those numbers are **not** the active NN_05 result and were not re-run here; they are recorded only because they are the measured reason the temperature was fixed at 0.5.

## Results, paired against the LightGBM baseline

Per-seed and seed-mean IC per cycle; the paired difference is the seed-mean NN IC minus the baseline IC on the same cycle, same names, same returns. Summary statistics follow `ICBacktestResult`'s definitions (population std, hit rate = share of cycles with IC > 0). The extra "ensemble of seeds" column is the IC of the five seeds' averaged score, reported for `NN_08`'s reference only. **Eight paired cycles is a thin basis**, far thinner than the 22-cycle full-history baseline; no win or loss is claimed from this table.

In [9]:
if rd.require_data():
    measured_keys = [c.quarter_start for c in MEASURED]
    nn_measured = sorted({datetime.fromisoformat(r["target"]).date() for r in RESULTS if r["ic"] is not None})
    assert nn_measured == measured_keys, "NN measured cycles must match the baseline's measured cycles exactly"
    assert all(set(r) >= {"seed", "target", "train_quarters", "validation_quarter", "scored_count", "ic", "best_val_loss", "final_train_loss"} for r in RESULTS)

    by_cycle_seed = {(r["target"], r["seed"]): r for r in RESULTS}
    base_ic = {q.isoformat(): baseline[q].ic for q in measured_keys}
    seed_mean, ensemble, paired = {}, {}, {}
    print(f"{'cycle':<12}{'baseline':>10}" + "".join(f"{'seed ' + str(s):>10}" for s in nc.SEEDS) + f"{'seed-mean':>11}{'ensemble':>10}{'NN - LGBM':>11}")
    for q in measured_keys:
        k = q.isoformat()
        ics = [by_cycle_seed[(k, s)]["ic"] for s in nc.SEEDS]
        seed_mean[k] = float(np.mean(ics))
        ids, rets, _ = UNIVERSE[q]
        mean_scores = np.mean([by_cycle_seed[(k, s)]["scores"] for s in nc.SEEDS], axis=0)
        ensemble[k] = ic_of(mean_scores, rets)
        paired[k] = seed_mean[k] - base_ic[k]
        print(f"{k:<12}{base_ic[k]:>+10.4f}" + "".join(f"{v:>+10.4f}" for v in ics) + f"{seed_mean[k]:>+11.4f}{ensemble[k]:>+10.4f}{paired[k]:>+11.4f}")

    def summary(values):
        v = np.array(values, dtype=float)
        return v.mean(), v.std(), (v > 0).mean()

    print()
    print(f"{'series':<28}{'mean IC':>9}{'std':>8}{'hit rate':>10}")
    b_mean, b_std, b_hit = summary([base_ic[q.isoformat()] for q in measured_keys])
    print(f"{'LightGBM baseline':<28}{b_mean:>+9.4f}{b_std:>8.4f}{b_hit:>10.3f}")
    per_seed_stats = {}
    for s in nc.SEEDS:
        m, sd, h = summary([by_cycle_seed[(q.isoformat(), s)]["ic"] for q in measured_keys])
        per_seed_stats[s] = (m, sd, h)
        print(f"{'NN seed ' + str(s):<28}{m:>+9.4f}{sd:>8.4f}{h:>10.3f}")
    sm_mean, sm_std, sm_hit = summary([seed_mean[q.isoformat()] for q in measured_keys])
    print(f"{'NN seed-mean IC':<28}{sm_mean:>+9.4f}{sm_std:>8.4f}{sm_hit:>10.3f}")
    en_mean, en_std, en_hit = summary([ensemble[q.isoformat()] for q in measured_keys])
    print(f"{'NN ensemble-of-seeds score':<28}{en_mean:>+9.4f}{en_std:>8.4f}{en_hit:>10.3f}")
    d = np.array([paired[q.isoformat()] for q in measured_keys])
    print(f"\npaired NN(seed-mean) - LightGBM over {len(d)} cycles: mean {d.mean():+.4f}, std {d.std():.4f}, "
          f"NN higher in {int((d > 0).sum())} of {len(d)} cycles")
    print(f"per-seed mean IC across seeds: mean {np.mean([v[0] for v in per_seed_stats.values()]):+.4f}, "
          f"min {min(v[0] for v in per_seed_stats.values()):+.4f}, max {max(v[0] for v in per_seed_stats.values()):+.4f}")
    best_steps = [r["best_step"] for r in RESULTS]
    print(f"early stopping: best validation step min {min(best_steps)}, median {int(np.median(best_steps))}, max {max(best_steps)}; "
          f"steps run min {min(r['steps'] for r in RESULTS)}, max {max(r['steps'] for r in RESULTS)}")
    print(f"fits whose best validation step is <= 50: {sum(1 for b in best_steps if b <= 50)} of {len(best_steps)}; "
          f"fits that ran to the {MAX_STEPS}-step cap: {sum(1 for r in RESULTS if r['steps'] == MAX_STEPS)}")
    expected_ic_calls = len(UNIVERSE) + sum(1 for r in RESULTS if r["ic"] is not None) + 1 + len(measured_keys)
    print(f"IC computations, all via spearman_correlation: {IC_CALLS} (expected {expected_ic_calls})")
    assert IC_CALLS == expected_ic_calls

cycle         baseline    seed 0    seed 1    seed 2    seed 3    seed 4  seed-mean  ensemble  NN - LGBM
2022-01-01     -0.0261   +0.0732   +0.0613   +0.0783   +0.0998   +0.0668    +0.0759   +0.0744    +0.1020
2022-04-01     -0.1900   -0.1182   -0.1573   -0.1370   -0.1676   -0.1308    -0.1422   -0.1501    +0.0478
2022-07-01     +0.0618   +0.0869   +0.1089   +0.1049   +0.1118   +0.1103    +0.1046   +0.1115    +0.0428
2022-10-01     +0.0583   +0.0909   +0.0195   +0.0707   +0.0555   +0.0800    +0.0633   +0.0719    +0.0050
2023-01-01     +0.0708   +0.0301   +0.0553   +0.0291   +0.0613   +0.1162    +0.0584   +0.0610    -0.0124
2023-04-01     +0.0335   +0.0878   +0.0348   +0.0714   +0.1456   +0.0975    +0.0874   +0.0952    +0.0539
2023-07-01     +0.0877   +0.0966   +0.0946   +0.1360   +0.1456   +0.0418    +0.1029   +0.1075    +0.0153
2023-10-01     +0.1240   +0.0513   +0.0475   +0.0642   +0.0579   -0.0026    +0.0436   +0.0482    -0.0803

series                        mean IC     std  hit rat

## Saved output

Per-cycle, per-seed records (including the NN scores, the baseline scores and the forward returns for the scored universe) are written for `NN_06`–`NN_08`, so later notebooks can pair and ensemble without reloading prices. Derived output, reproducible bit for bit on CPU.

In [10]:
if rd.require_data():
    import json

    OUT_DIR = nc.NEURAL_RANKER_DIR / "outputs"
    OUT_DIR.mkdir(exist_ok=True)
    out_path = OUT_DIR / "NN_05_walkforward.json"
    payload = {
        "notebook": "NN_05_walkforward.ipynb",
        "git_commit": rd.git_commit(), "config_identity": cfg.identity(), "slice_fingerprint": bundle.fingerprint,
        "device": str(DEVICE), "seeds": list(nc.SEEDS),
        "protocol": {"lr": LR, "weight_decay": WEIGHT_DECAY, "max_steps": MAX_STEPS, "eval_every": EVAL_EVERY, "patience": PATIENCE,
                     "temperature": TEMPERATURE, "target": "softmax(((2**y - 1) / (2**9 - 1)) / 0.5)",
                     "holdout": "most recent included training quarter"},
        "universe": {q.isoformat(): {"symbols": [i.symbol for i in ids], "returns": rets, "baseline_scores": base}
                     for q, (ids, rets, base) in UNIVERSE.items()},
        "baseline_ic": base_ic,
        "results": RESULTS,
    }
    out_path.write_text(json.dumps(payload))
    print("wrote", out_path, f"({out_path.stat().st_size / 1e6:.2f} MB)")

wrote /Users/gavinhussey/Downloads/AtlasQuant/neural_ranker/outputs/NN_05_walkforward.json (1.73 MB)


## Point-in-time check

Stated explicitly, per `CLAUDE.md` and `BUILD_SPEC.md` §6, and asserted inside every fit rather than assumed:

1. **No target-quarter or future-quarter row in training or validation.** Every training and validation quarter is strictly before its target; the target quarter itself is never in the included set.
2. **The validation quarter is inside the training window** — the most recent quarter the production window includes, never at or after the target.
3. **Every label used for training or validation was available strictly before the target's training cutoff** (`label_available_at < cutoff`, the same strict rule `build_training_dataset` applies).
4. **Normalization and the input transform** are the per-cycle policy audited in `NN_02`/`NN_03` and the elementwise map verified in `NN_04`; no statistic crosses cycles.
5. **Scoring** uses only the target cycle's own tensors and `m`; the forward returns are realized after the cycle and enter only the IC, never the model.

In [11]:
if rd.require_data():
    checked = 0
    for r in RESULTS:
        target = datetime.fromisoformat(r["target"]).date()
        target_cycle = next(c for c in TRAINED if c.quarter_start == target)
        cutoff = datetime.combine(target_cycle.cutoff, _time.min)
        used = [datetime.fromisoformat(q).date() for q in r["train_quarters"]] + [datetime.fromisoformat(r["validation_quarter"]).date()]
        assert target not in used and all(q < target for q in used)
        assert all(ALL[q].label_available_at < cutoff for q in used)
        assert datetime.fromisoformat(r["validation_quarter"]).date() == max(used)
        checked += 1
    print(f"records checked: {checked}; training/validation quarters before target, labels available before cutoff, validation = most recent window quarter: True")
    print("point-in-time checks 1-5: passed")

records checked: 45; training/validation quarters before target, labels available before cutoff, validation = most recent window quarter: True
point-in-time checks 1-5: passed


## What this established

Every number below was printed by a cell above in this executed run. Because every fit ran on CPU, the ICs, losses and scores are bit-reproducible; only the timings vary between runs. **This is raw walk-forward output; no win or loss against LightGBM is claimed. `NN_06` is the decisive ablation.**

**The training objective is fixed as a protocol decision.** `NN_04` implemented the literal `softmax(2**y − 1)` and measured that it puts mass 1.0000 on grade 9 and 0 on every other grade (1 grade of 10; entropy 4.9972). The graded target with temperature 1.0 was measured in the first version of this notebook to be too close to uniform (entropy 7.2436 against log N = 7.3085, 0.0649 nats of attainable improvement per list). The active target is `softmax(((2**y − 1)/(2**9 − 1)) / 0.5)` — **temperature 0.5, set once, never tuned, shared by `NN_06`–`NN_08`**. On 2023-10-01 it gives every grade non-zero mass (10 of 10): 0.0584 on grade 0 (167 names), 0.0516 .. 0.0845 on grades 1–7, 0.1394 on grade 8, 0.3823 on grade 9; entropy 6.9843, **0.2593 nats below the temperature-1.0 target**, with 0.3242 nats of attainable improvement per list (five times the temperature-1.0 room).

**Pairing is exact.** With the complete real price history loaded (59.8 s this run), the baseline IC was recomputed bit-for-bit from `bundle.ic_result`'s stored Materials-excluded ranked candidates on all 8 measured cycles; scored counts 1,370 .. 1,401 equal the stored counts; 0 Materials names in any scored set; 72–73 candidates rejected per cycle. The NN was scored on exactly those names and those returns. 57 IC values were reported in this notebook, every one through `spearman_correlation`; the diagnostic below uses the same function. The NN's measured cycles are the baseline's 8, exactly.

**Inputs and device.** 14 cycles on `cpu` (8 threads), all inputs finite after the placeholder step, mask equal to the original `NaN` locations, 25,684 placeholder entries in total. The first trained cycle, seed 0, refitted: IC +0.073248 both times, best validation loss 7.285092 both times, scores bit-identical.

**The walk-forward.** 9 trained cycles × 5 seeds = 45 fits, 8 cycles with a measured IC (the 2024-01-01 fits have no next quarter and record `ic = None`, as the baseline does). Training windows of 3 to 11 quarters plus 1 validation quarter; production `build_training_dataset` rows matched the per-quarter tensors row for row on every cycle. 243 s in total, mean 5.4 s per fit (2.2 to 9.4 s), growing with the window; timings are the one quantity that varies between runs.

**Early stopping still fires immediately — reported plainly.** The best validation loss came at step 10 to 50 (median 20) on 45 of 45 fits; every fit stopped by patience at step 210 to 250; 0 fits reached the 1,000-step cap. The sharper target did not change this. The diagnostic (seed 0, every cycle, 1,000 steps without early stopping, 228 s) shows why: the mean validation loss is 7.28927 at step 10, 7.28736 at step 20, then rises monotonically — 7.30342 at 50, 7.34980 at 100, 7.40859 at 200, 7.44498 at 1,000 — while the mean training loss falls from 7.25884 to 7.01802. Validation ranking quality, Spearman(score, grade) on the held-out quarter, averages +0.0628 at step 20, +0.0266 at 100, +0.0513 at 500 and +0.0430 at 1,000, and its per-cycle maximum lands anywhere from step 10 to step 890. Under this protocol (full-batch AdamW, lr 1e-3, weight decay 1e-2, dropout 0.3) the network learns what transfers across quarters within its first tens of steps and then fits the training quarters; the loss-minimum rule picks those early models, and longer training does not improve validation ranking on average. This is a property of the optimization protocol, not of the target's sharpness; it is recorded, not tuned, and `NN_06` compares both arms under it.

**Per-cycle IC on the 8 measured cycles.**

| cycle | LightGBM | seed 0 | seed 1 | seed 2 | seed 3 | seed 4 | seed-mean | ensemble | NN − LGBM |
|---|---|---|---|---|---|---|---|---|---|
| 2022-01-01 | −0.0261 | +0.0732 | +0.0613 | +0.0783 | +0.0998 | +0.0668 | +0.0759 | +0.0744 | +0.1020 |
| 2022-04-01 | −0.1900 | −0.1182 | −0.1573 | −0.1370 | −0.1676 | −0.1308 | −0.1422 | −0.1501 | +0.0478 |
| 2022-07-01 | +0.0618 | +0.0869 | +0.1089 | +0.1049 | +0.1118 | +0.1103 | +0.1046 | +0.1115 | +0.0428 |
| 2022-10-01 | +0.0583 | +0.0909 | +0.0195 | +0.0707 | +0.0555 | +0.0800 | +0.0633 | +0.0719 | +0.0050 |
| 2023-01-01 | +0.0708 | +0.0301 | +0.0553 | +0.0291 | +0.0613 | +0.1162 | +0.0584 | +0.0610 | −0.0124 |
| 2023-04-01 | +0.0335 | +0.0878 | +0.0348 | +0.0714 | +0.1456 | +0.0975 | +0.0874 | +0.0952 | +0.0539 |
| 2023-07-01 | +0.0877 | +0.0966 | +0.0946 | +0.1360 | +0.1456 | +0.0418 | +0.1029 | +0.1075 | +0.0153 |
| 2023-10-01 | +0.1240 | +0.0513 | +0.0475 | +0.0642 | +0.0579 | −0.0026 | +0.0436 | +0.0482 | −0.0803 |

**Summary over the 8 cycles.** LightGBM: mean IC +0.0275, std 0.0916, hit rate 0.750. NN per seed: +0.0498, +0.0331, +0.0522, +0.0637, +0.0474 (std 0.0670 .. 0.0940; hit rate 0.750 .. 0.875). NN seed-mean IC: **+0.0492**, std 0.0751, hit rate 0.875. Ensemble-of-seeds score: +0.0524, std 0.0793, hit rate 0.875. Paired NN(seed-mean) − LightGBM: **mean +0.0218, std 0.0506, NN higher in 6 of 8 cycles**. Against the superseded temperature-1.0 run (seed-mean +0.0445, paired +0.0170, same 6 of 8), the sharper target moved the seed-mean IC by +0.005 — inside the seed spread.

**How to read that, and what is not claimed.** The spread of the per-seed mean IC across seeds (+0.0331 to +0.0637) is as wide as the paired difference itself, and within single cycles the seed spread reaches 0.15 (2023-04-01: +0.0348 to +0.1456). Eight paired cycles is far thinner than the 22-cycle full-history baseline; no significance test is run here and no win or loss is concluded. What the table does establish is an IC series on the baseline's exact cycles, names and returns, under a fixed protocol, with the seed noise visible alongside it.

**Point-in-time, checked on all 45 records.** Every training and validation quarter is strictly before its target, every label used was available strictly before the target's cutoff, the validation quarter is the most recent quarter of the production window, and the target quarter never appears in the window. The diagnostic looked only at validation quarters inside the window.

**Saved for `NN_06`–`NN_08` (overwritten this run).** `neural_ranker/outputs/NN_05_walkforward.json`, gitignored: per-fit records with NN scores, the protocol including `temperature: 0.5`, plus the baseline scores and forward returns for each cycle's scored universe.

**What `NN_06` needs from here.** The identical protocol with `use_film=False` as the paired twin (same seeds, same windows, same names, temperature 0.5, CPU), the fact that both arms will be 10–50-step models under this protocol, and the saved output file for pairing and for a bit-for-bit cross-check of the FiLM arm.